# Example set - Builders Temeplate include

In [ ]:
# ===== Shared setup: every example cell depends on this one, run it first =====
import numpy as np
import matplotlib.pyplot as plt

from admiser import OCPSolver

# There is exactly one solve entry point: OCPSolver(problem).solve(...)
# Whether that is a single solve or an eps -> 0 continuation, and the number of
# rounds and shrink ratio, are declared in the PROBLEM DEFINITION FILE:
#     problem.set_transcription(mode="continuation", n_rounds=4, shrink=0.1)
# Nothing here has to branch on it, and the result dictionary has the same shape
# either way (the "rounds" field simply has length 1 in single mode).


def solve_modes(EX, modes=("single", "continuation"), **solve_kw):
    """
    Solve the same problem once per requested mode, for comparison.

    The mode parameters (n_rounds / shrink) are taken from the set_transcription()
    call in the problem file; only `mode` is overridden here, and the original
    setting is restored afterwards. solve() never mutates the problem, so calling
    it repeatedly is safe.
    """
    p = EX.problem
    default = dict(p.transcription)
    out = {}
    try:
        for m in modes:
            cfg = dict(default); cfg["mode"] = m
            p.set_transcription(**cfg)
            print(f"\n########## mode: {m} ##########")
            out[m] = OCPSolver(p).solve(**solve_kw)
    finally:
        p.set_transcription(**default)
    return out


def report_result(tag, res):
    """
    Print the results of one mode; for a multi-round solve, list eps/gamma and the
    true violation for every round.
    """
    r = res["scipy_result"]
    print(f"\n=== {tag} ===")
    print(f"J*     = {res['J_opt']}")
    print(f"status = {r.status}  ({r.message})")
    if res.get("theta_opt") is not None:
        print(f"theta* = {res['theta_opt']}")
    if res.get("eq_resid") is not None:
        print(f"max|eq|  = {np.max(np.abs(res['eq_resid'])):.3e}   (should be ~ 0)")
    if res.get("ineq_resid") is not None:
        print(f"min ineq = {np.min(res['ineq_resid']):.3e}   (should be >= 0)")
    if res.get("path_viol") is not None:
        print(f"max h(t) = {np.max(res['path_viol']):+.3e}   (true violation of the original constraint, should be <= 0)")
    if len(res.get("rounds", [])) > 1:
        print("  per round:")
        for i, hh in enumerate(res["rounds"]):
            print(f"    [{i+1}] eps={min(hh['eps']):.1e}  gamma={min(hh['gamma']):.3e}  "
                  f"J={hh['J_opt']:+.10g}  max h(t)={hh['max_path_viol']:+.3e}  "
                  f"status={hh['status']}  nit={hh['nit']}")


#: How the two modes are told apart in the plots: solid = single, dashed = continuation
STYLE = {"single": dict(ls="-", lw=1.6, alpha=1.0),
         "continuation": dict(ls="--", lw=1.6, alpha=0.9)}


## No system parameters, pure OCP

In [ ]:
# solve_my_robot_with_param.py
from admiser.examples.my_robot_problem import problem, N, dt, x0, xT

def main():
    solver = OCPSolver(problem)
    result = solver.solve(maxiter=500, ftol=1e-9, disp=True)

    U_opt     = result["U_opt"]
    theta_opt = result["theta_opt"]
    X_opt     = result["X_opt"]
    t_opt     = result["t_opt"]
    J_opt     = result["J_opt"]
    eq_err  = result["eq_resid"]      # equality residual G(z), should be ~ 0
    ineq_err= result["ineq_resid"]    # inequality residual C(z), should be >= 0

    print("\n=== report ===")
    print("theta_opt:", theta_opt)
    print("J*:", J_opt)
    if eq_err is not None:
        print("equality constraint residual (includes ∫L·dt terms):")
        print(eq_err)
    if ineq_err is not None:
        print("inequality constraint residual (includes ∫L·dt terms):")
        print(ineq_err)

    # state trajectories
    plt.figure()
    plt.plot(X_opt[:,0], X_opt[:,1], '-o', ms=2, label='trajectory')
    plt.plot(x0[0], x0[1], 's', label='start'); plt.plot(xT[0], xT[1], 'x', label='target')
    plt.axis('equal'); plt.legend(); plt.title('path'); plt.xlabel('x1'); plt.ylabel('x2')

    # controls
    U = U_opt.reshape(N, 2)
    tc = np.linspace(0.0, N*dt, N, endpoint=False) + 0.5*dt
    plt.figure(); plt.step(tc, U[:,0], where='mid', label='u1'); plt.step(tc, U[:,1], where='mid', label='u2')
    plt.legend(); plt.title('controls'); plt.xlabel('t')

    plt.show()

if __name__ == "__main__":
    main()


## Pure OCP - initial guess sensitive

In [ ]:
# solve_my_ocp_problem_tpl.py
from admiser.examples.my_bitumen_pyrolysis import problem, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-9, disp=True)

    U_opt  = res["U_opt"]              # shape = (N*nu,)
    theta  = res.get("theta_opt", None)
    J_opt  = res["J_opt"]
    X_opt  = res["X_opt"]              # shape = (N+1, nx)
    t_opt  = res["t_opt"]              # shape = (N+1,)
    eq_res = res.get("eq_resid", None)   # equality residual G(z), should be ~ 0
    in_res = res.get("ineq_resid", None) # inequality residual C(z), should be >= 0

    print("\n=== results ===")
    print("J* =", J_opt)
    if theta is not None:
        print("theta* =", theta)
    if eq_res is not None:
        print("eq residual:", eq_res)
    if in_res is not None:
        print("ineq residual:", in_res)

    # states
    plt.figure()
    for i in range(X_opt.shape[1]):
        plt.plot(t_opt, X_opt[:, i], label=f"x{i+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend()

    # controls: piecewise constant over N segments; reshape the flat U_opt
    nu = problem.nu
    U_mat = U_opt.reshape(N, nu) if U_opt.ndim == 1 else U_opt
    tc = np.linspace(0.0, np.sum(np.diff(t_opt)), N)  # equals linspace(0, T, N) for constant dt
    plt.figure()
    for j in range(nu):
        plt.step(tc, U_mat[:, j], where='pre', label=f"u{j+1}")
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


# System parameters are included

## Feedback control

In [ ]:
from admiser.examples.my_policy_param_problem import problem, N, dt, policy_u

def main():
    solver = OCPSolver(problem)
    result = solver.solve(maxiter=600, ftol=1e-9, disp=True)

    U_opt     = result["U_opt"]        # length 0 here, since nu = 0
    theta_opt = result["theta_opt"]    # optimal [z1..z4]
    X_opt     = result["X_opt"]
    t_opt     = result["t_opt"]
    J_opt     = result["J_opt"]

    print("\n=== report ===")
    print("theta_opt [z1,z2,z3,z4] =", theta_opt)
    print("J* =", J_opt)

    # reconstruct the u(t) trajectory from the optimal theta, for plotting
    U_series = np.array([ float(policy_u(x, theta_opt)) for x in X_opt ])

    # states
    plt.figure()
    plt.plot(t_opt, X_opt[:,0], label='x1')
    plt.plot(t_opt, X_opt[:,1], label='x2')
    plt.xlabel('t'); plt.ylabel('states'); plt.title('States'); plt.legend()

    # controls (left endpoint of each segment)
    tc = np.linspace(0.0, N*dt, N+1)
    plt.figure()
    plt.step(tc, U_series, where='pre')
    plt.xlabel('t'); plt.ylabel('u(t)'); plt.title('Control from policy θ')

    plt.show()

if __name__ == "__main__":
    main()

## Recursive control

In [ ]:
# solve_ex_882_theta_zeta.py
from admiser.examples.my_ex_882_zeta_param import problem, N, dt, u_max

def main():
    solver = OCPSolver(problem)
    # two equality constraints, solved with SLSQP
    res = solver.solve(maxiter=800, ftol=1e-9, disp=True)

    U_opt     = res["U_opt"]        # shape=(N,)
    theta_opt = res["theta_opt"]    # [ζ1, ζ2]
    J_opt     = res["J_opt"]
    X_opt     = res["X_opt"]
    t_opt     = res["t_opt"]

    z1, z2 = theta_opt
    print("\n=== optimal solution (key quantities) ===")
    print(f"zeta1 = {z1:.6f},  zeta2 = {z2:.6f}   (the objective is -zeta2 = {-z2:.9f})")
    print(f"J* = {J_opt:.9f}")
    print(f"u_max = {u_max}")

    plt.figure()
    plt.plot(t_opt, X_opt[:,0], label='x1(t)')
    plt.plot(t_opt, X_opt[:,1], label='x2(t)')
    plt.xlabel('t'); plt.ylabel('state'); plt.title('States'); plt.legend()

    tc = np.linspace(0.0, N*dt, N)
    plt.figure()
    plt.step(tc, U_opt, where='pre')
    plt.axhline(0.0, color='k', lw=0.5)
    plt.axhline(u_max, color='gray', ls='--', lw=0.8)
    plt.xlabel('t'); plt.ylabel('u'); plt.title('Control (piecewise-constant)')

    plt.show()

if __name__ == "__main__":
    main()


# Continuous state and control inequality constraints

## Simple example - continuous state inequality

In [ ]:
# solve_state_constrained.py
from admiser.examples import my_state_constrained_problem as EX
N, dt, T = EX.N, EX.dt, EX.T

# The solve mode is declared in my_state_constrained_problem.py; list here the
# modes you want to compare. To run only the one the problem file declares, use
# MODES = (EX.problem.transcription["mode"],)
MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=1000, ftol=1e-12)
    for tag, res in results.items():
        report_result(tag, res)

    # ---- state trajectories: colour by state component, line style by solve mode ----
    plt.figure(figsize=(8, 4.5))
    for tag, res in results.items():
        for i, name in enumerate(("x1", "x2")):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"{name} ({tag})", **STYLE[tag])
    t_ref = next(iter(results.values()))["t_opt"]
    plt.plot(t_ref, 8.0 * (t_ref - 0.5) ** 2 - 0.5, "k:", lw=1.2,
             label="x2 upper bound 8(t-0.5)^2-0.5")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend(fontsize=8)

    # ---- true violation of the path constraint ----
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        h_ = [EX.hfun(t_[k], X_[k], None, None) for k in range(len(t_))]
        plt.plot(t_, h_, label=f"h(t) ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("path constraint h(t) <= 0  (positive means violated)")
    plt.legend(fontsize=8)

    # ---- controls (piecewise constant) ----
    tc = np.linspace(0.0, N * dt, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        plt.step(tc, res["U_opt"], where="mid", label=f"u ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.5)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Control (piecewise-constant)")
    plt.legend(fontsize=8)

    plt.show()


if __name__ == "__main__":
    main()


## Continuous state and control constraint

In [ ]:
# solve_state_control_constraint.py
from admiser.examples import my_state_control_constraint as EX
N, dt, T = EX.N, EX.dt, EX.T

MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=2000, ftol=1e-9)
    for tag, res in results.items():
        report_result(tag, res)

    nu = EX.problem.nu

    # ---- states ----
    plt.figure(figsize=(8, 4.5))
    for tag, res in results.items():
        for i in range(res["X_opt"].shape[1]):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"x{i+1} ({tag})", **STYLE[tag])
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend(fontsize=8)

    # ---- true violation of the mixed state-control constraint h = u + x1/6 <= 0 ----
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        U_ = res["U_opt"].reshape(N, nu)
        h_ = [EX.h(t_[k], X_[k], U_[min(k, N - 1)], None) for k in range(len(t_))]
        plt.plot(t_, h_, label=f"h(t) ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("path constraint h = u + x1/6 <= 0  (positive means violated)")
    plt.legend(fontsize=8)

    # ---- controls (piecewise constant) ----
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        U_mat = res["U_opt"].reshape(N, nu)
        for j in range(nu):
            plt.step(tc, U_mat[:, j], where="mid", color=f"C{j}",
                     label=f"u{j+1} ({tag})", **STYLE[tag])
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend(fontsize=8)

    plt.show()


if __name__ == "__main__":
    main()


## More commplicate problem - kobe port

In [ ]:
# solve_six_state.py (port_kobe)
from admiser.examples import my_port_kobe as EX
T, N, dt = EX.T, EX.N, EX.dt

MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=2000, ftol=1e-8)
    for tag, res in results.items():
        report_result(tag, res)

    # ---- states ----
    plt.figure(figsize=(10, 5))
    for tag, res in results.items():
        for i in range(6):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"x{i+1} ({tag})", **STYLE[tag])
    t_ref = next(iter(results.values()))["t_opt"]
    for lvl, ls in ((2.5, "k--"), (-2.5, "k--"), (1.0, "k-."), (-1.0, "k-.")):
        plt.plot(t_ref, lvl * np.ones_like(t_ref), ls, lw=0.8)
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States (dashed = x4 bounds, dash-dot = x5 bounds)")
    plt.legend(ncol=3, fontsize=7)

    # ---- true violation of the four path constraints ----
    plt.figure(figsize=(10, 3.2))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        for j, hf in enumerate(EX.PATH_INEQS):
            h_ = [hf(t_[k], X_[k], None, None) for k in range(len(t_))]
            plt.plot(t_, h_, color=f"C{j}", label=f"h{j+1} ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("four path constraints h <= 0  (positive means violated)")
    plt.legend(ncol=4, fontsize=7)

    # ---- controls (piecewise constant) ----
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(10, 3.5))
    for tag, res in results.items():
        U = res["U_opt"].reshape(N, 2)
        plt.step(tc, U[:, 0], where="mid", color="C0", label=f"u1 ({tag})", **STYLE[tag])
        plt.step(tc, U[:, 1], where="mid", color="C1", label=f"u2 ({tag})", **STYLE[tag])
    for lvl, ls in ((2.83374, "--"), (-2.83374, "--"), (0.71265, "-."), (-0.80865, "-.")):
        plt.axhline(lvl, color="gray", ls=ls, lw=0.8)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend(fontsize=7)

    plt.show()


if __name__ == "__main__":
    main()


## Optimal Euler buckling beam - continuous inequality and system parameters

In [ ]:
# solve_z1z2_problem.py (euler buckling beam)
from admiser.examples import my_euler_buckling_beam as EX
T, N, dt = EX.T, EX.N, EX.dt

MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=2000, ftol=1e-9)
    for tag, res in results.items():
        report_result(tag, res)
        z1, z2 = res["theta_opt"]
        print(f"  z1* = {z1:.6f},  z2* = {z2:.6f}")

    # ---- states ----
    plt.figure(figsize=(8, 4.5))
    for tag, res in results.items():
        for i in range(3):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"x{i+1} ({tag})", **STYLE[tag])
    plt.axhline(0.5, color="k", ls=":", lw=1.2, label="x3 ≥ 0.5")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend(fontsize=8)

    # ---- true violation of the path constraint h = 0.5 - x3 <= 0 ----
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        h_ = [EX.hfun(t_[k], X_[k], None, None) for k in range(len(t_))]
        plt.plot(t_, h_, label=f"h(t) ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("path constraint h = 0.5 - x3 <= 0  (positive means violated)")
    plt.legend(fontsize=8)

    # ---- controls (piecewise constant) ----
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        plt.step(tc, res["U_opt"], where="mid", label=f"u ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.5)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Control"); plt.legend(fontsize=8)

    plt.show()


if __name__ == "__main__":
    main()


# Bang-bang control

In [ ]:
# solve_lin2x2.py
from admiser.examples.my_bang_bang import problem, T, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-8, disp=True)

    U_opt    = res["U_opt"]         # shape = (2N,)
    X_opt    = res["X_opt"]         # shape = (N+1, 2)
    t_opt    = res["t_opt"]

    print("\n=== results ===")
    print("J* =", res["J_opt"])
    if res["eq_resid"]   is not None: print("eq residual:",   res["eq_resid"])
    if res["ineq_resid"] is not None: print("ineq residual:", res["ineq_resid"])

    # state trajectories
    plt.figure(figsize=(7,4))
    plt.plot(t_opt, X_opt[:,0], label='x1')
    plt.plot(t_opt, X_opt[:,1], label='x2')
    plt.xlabel('t'); plt.ylabel('state'); plt.title('States'); plt.legend()

    # controls (step plot)
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5*dt
    U = U_opt.reshape(N, problem.nu)
    plt.figure(figsize=(7,3))
    plt.step(tc, U[:,0], where='mid', label='u1')
    plt.step(tc, U[:,1], where='mid', label='u2')
    plt.axhline( 10.0, color='gray', ls='--', lw=0.8)
    plt.axhline(-10.0, color='gray', ls='--', lw=0.8)
    plt.xlabel('t'); plt.ylabel('u'); plt.title('Controls'); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


# Free Terminal Time

## From Visual MISER — minimum time via CPET


In [ ]:
# solve_free_terminal_time.py -- minimum time via the time-scaling transform (CPET)
from admiser.examples import my_free_terminal_time as EX
N = EX.N


def main():
    res = OCPSolver(EX.problem).solve(maxiter=3000, ftol=1e-10)
    report_result("minimum time (CPET)", res)

    U_opt = res["U_opt"]
    X_opt = res["X_opt"]
    t_opt = res["t_opt"]        # non-uniform: the optimised knot positions
    tau   = res["tau_opt"]

    print(f"\nminimum time T* = {res['T_opt']:.9f}")
    print("segment durations tau* =", np.array2string(tau, precision=4, suppress_small=True))
    print("(durations sitting at their lower bound are segments the optimiser dropped)")

    # ---- states against real time ----
    plt.figure(figsize=(8, 4))
    for i, lab in enumerate(("x1 heading", "x2", "x3")):
        plt.plot(t_opt, X_opt[:, i], marker="o", ms=3, label=lab)
    plt.axhline(0.0, color="k", lw=0.5)
    plt.xlabel("t"); plt.ylabel("state")
    plt.title(f"States (markers show the optimised knots), T* = {res['T_opt']:.6f}")
    plt.legend()

    # ---- control, on the non-uniform grid ----
    # t_opt already holds the true knot positions, so plot the steps against it
    # rather than rebuilding a uniform linspace.
    nu = EX.problem.nu
    U_mat = U_opt.reshape(N, nu)
    plt.figure(figsize=(8, 3))
    for j in range(nu):
        plt.step(t_opt[:-1], U_mat[:, j], where="post", label=f"u{j+1}")
    plt.axhline(0.0, color="k", lw=0.5)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Control (piecewise constant, uneven segments)")
    plt.legend()

    # ---- how long each segment ended up ----
    plt.figure(figsize=(8, 2.6))
    plt.bar(range(1, N + 1), tau)
    plt.xlabel("segment k"); plt.ylabel(r"$\tau_k$")
    plt.title("Optimised segment durations")

    plt.show()


if __name__ == "__main__":
    main()


## From Teo CPET — minimum time via CPET


In [ ]:
# solve_ratio_control.py -- minimum time via the time-scaling transform (CPET)
from admiser.examples import my_free_terminal_time2 as EX
N = EX.N


def main():
    res = OCPSolver(EX.problem).solve(maxiter=3000, ftol=1e-10)
    report_result("minimum time (CPET)", res)

    U_opt = res["U_opt"]
    X_opt = res["X_opt"]
    t_opt = res["t_opt"]
    tau   = res["tau_opt"]

    print(f"\nminimum time T* = {res['T_opt']:.9f}")
    print("terminal state x(T) =", X_opt[-1], "   (target [1.25, 1.0])")
    print("segment durations tau* =", np.array2string(tau, precision=4, suppress_small=True))

    # ---- states against real time ----
    plt.figure(figsize=(8, 4))
    plt.plot(t_opt, X_opt[:, 0], marker="o", ms=3, label="x1 concentration")
    plt.plot(t_opt, X_opt[:, 1], marker="o", ms=3, label="x2 volume")
    plt.axhline(1.25, color="C0", ls=":", lw=1.0)
    plt.axhline(1.00, color="C1", ls=":", lw=1.0)
    plt.xlabel("t"); plt.ylabel("state")
    plt.title(f"States (dotted = terminal targets), T* = {res['T_opt']:.6f}")
    plt.legend()

    # ---- controls, on the non-uniform grid ----
    U_mat = U_opt.reshape(N, 2)
    plt.figure(figsize=(8, 3))
    plt.step(t_opt[:-1], U_mat[:, 0], where="post", label="u1 in [0, 0.03]")
    plt.step(t_opt[:-1], U_mat[:, 1], where="post", label="u2 in [0, 0.01]")
    plt.xlabel("t"); plt.ylabel("control"); plt.title("Controls (uneven segments)")
    plt.legend()

    # ---- how long each segment ended up ----
    plt.figure(figsize=(8, 2.6))
    plt.bar(range(1, N + 1), tau)
    plt.xlabel("segment k"); plt.ylabel(r"$\tau_k$")
    plt.title("Optimised segment durations")

    plt.show()


if __name__ == "__main__":
    main()


# complicate problem

## Two drugs cancer

In [ ]:
# solve_my_ocp_problem_tpl.py
from admiser.examples.my_medical1 import problem, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-9, disp=True)

    U_opt  = res["U_opt"]              # shape = (N*nu,)
    theta  = res.get("theta_opt", None)
    J_opt  = res["J_opt"]
    X_opt  = res["X_opt"]              # shape = (N+1, nx)
    t_opt  = res["t_opt"]              # shape = (N+1,)
    eq_res = res.get("eq_resid", None)   # equality residual G(z), should be ~ 0
    in_res = res.get("ineq_resid", None) # inequality residual C(z), should be >= 0

    print("\n=== results ===")
    print("J* =", J_opt)
    if theta is not None:
        print("theta* =", theta)
    if eq_res is not None:
        print("eq residual:", eq_res)
    if in_res is not None:
        print("ineq residual:", in_res)

    # states
    plt.figure()
    plt.plot(t_opt, X_opt[:, 3], label=f"x{3+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("Normal cells"); plt.legend()

    plt.figure()
    plt.plot(t_opt, X_opt[:, 0] + X_opt[:, 1] + X_opt[:, 2], label=f"x{3+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("Total tumor cells"); plt.legend()

    # controls: piecewise constant over N segments; reshape the flat U_opt
    nu = problem.nu
    U_mat = U_opt.reshape(N, nu) if U_opt.ndim == 1 else U_opt
    tc = np.linspace(0.0, np.sum(np.diff(t_opt)), N)  # equals linspace(0, T, N) for constant dt
    plt.figure()
    for j in range(nu):
        plt.step(tc, U_mat[:, j], where='pre', label=f"u{j+1}")
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


## SRI

In [ ]:
# solve_covid_seir_ocp.py
from admiser.examples.my_medical2 import problem, N, dt, T, u1_max, u2_max, I_cap, V_budget

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=3000, ftol=1e-6, disp=True)

    print("\n=== results ===")
    print("J* =", res["J_opt"])
    if "term_err" in res and res["term_err"] is not None:
        print("eq residual:", res["term_err"])
    if "ineq_resid" in res and res["ineq_resid"] is not None:
        print("ineq residual(s):", res["ineq_resid"])

    U  = res["U_opt"].reshape(N, -1)   # (N,2)
    t  = res["t_opt"]
    X  = res["X_opt"]                  # (N+1,4): [S,E,I,R]
    S,E,I,R = X[:,0], X[:,1], X[:,2], X[:,3]

    # --- states ---
    plt.figure()
    plt.plot(t, S, label="S")
    plt.plot(t, E, label="E")
    plt.plot(t, I, label="I")
    plt.plot(t, R, label="R")
    plt.axhline(I_cap, color='r', ls='--', lw=0.8, label="I cap")
    plt.xlabel("Day"); plt.ylabel("Fraction")
    plt.title("SEIR States")
    plt.legend()

    # --- controls ---
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5*dt
    plt.figure()
    plt.step(tc, U[:,0], where='mid', label="u1: vaccination")
    plt.axhline(0.0, color='k', lw=0.5)
    plt.axhline(u1_max, color='gray', ls='--', lw=0.8)
    plt.step(tc, U[:,1], where='mid', label="u2: contact reduction")
    plt.axhline(u2_max, color='gray', ls='--', lw=0.8)
    plt.xlabel("Day"); plt.ylabel("Control")
    plt.title("Optimal Controls")
    plt.legend()

    # --- cumulative vaccine usage, approximating int u1*S dt ---
    v_consumed = np.trapezoid(U[:,0] * 0.5*(S[:-1] + S[1:]), dx=dt)
    print(f"\nEstimated ∫ u1*S dt = {v_consumed:.4f} (budget ≤ {V_budget})")

    plt.show()

if __name__ == "__main__":
    main()


### TODO - Problem Scale, Time Scaling
- linear growth condition, Lipschitz condition, existence and uniqueness of the solution